In [9]:
!mkdir -p /content/train/
!find "/content/drive/Shareddrives/Computer Vision/output/train/" -mindepth 1 -maxdepth 1 -exec cp -r -t /content/train/ {} +


In [10]:
!mkdir -p /content/generated/
!find "/content/drive/Shareddrives/Computer Vision/generation_output/" -mindepth 1 -maxdepth 1 -exec cp -r -t /content/generated/ {} +


In [1]:
!pip install torchmetrics[image] torchvision tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 54.6 MB/s eta 0:00:00


In [11]:
import os
import glob
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
from tqdm import tqdm

REAL_DATA_DIR = '/content/train'
GEN_DATA_DIR = '/content/generated'
BATCH_SIZE = 50
NUM_WORKERS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE = 32

LPIPS_PAIRS_PER_CLASS = 1000


In [12]:
class ImageFolderDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = sorted(glob.glob(os.path.join(root_dir, '**', '*.*'), recursive=True))
        self.image_paths = [p for p in self.image_paths if p.lower().endswith(('.png', '.jpg', '.jpeg'))]

        if len(self.image_paths) == 0:
            print(f"Warning: No images found in {root_dir}")
        else:
            print(f"Found {len(self.image_paths)} images in {root_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image


In [13]:
def verify_size(img, target_size):
    w, h = img.size
    if w != target_size or h != target_size:
        raise ValueError(f"Found generated image with wrong size: {w}x{h}. Expected: {target_size}x{target_size}")
    return img

def get_dataloader(root_dir, batch_size, target_size, is_real_data=False):

    transform_list = []

    if is_real_data:
        transform_list.append(transforms.Resize((target_size, target_size), antialias=True))
    else:
        transform_list.append(transforms.Lambda(lambda img: verify_size(img, target_size)))

    transform_list.append(transforms.ToTensor())

    transform = transforms.Compose(transform_list)
    dataset = ImageFolderDataset(root_dir, transform=transform)

    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)



In [14]:
def calculate_lpips_diversity(gen_dir, device):

    print(f"\n=== Calculating Per-Class LPIPS Diversity (Sampling {LPIPS_PAIRS_PER_CLASS} pairs) ===")

    lpips_metric = LearnedPerceptualImagePatchSimilarity(net_type='vgg', normalize=True).to(device)

    lpips_transform = transforms.Compose([
        transforms.Resize((64, 64), antialias=True),
        transforms.ToTensor()
    ])

    subfolders = sorted([d for d in os.listdir(gen_dir) if os.path.isdir(os.path.join(gen_dir, d))])
    all_scores = []

    for class_folder in subfolders:
        class_path = os.path.join(gen_dir, class_folder)
        images = sorted(glob.glob(os.path.join(class_path, '*.*')))
        images = [i for i in images if i.lower().endswith(('.png', '.jpg'))]

        if len(images) < 2:
            continue

        current_scores = []

        for _ in range(LPIPS_PAIRS_PER_CLASS):
            idx1, idx2 = torch.randint(0, len(images), (2,))

            img1 = lpips_transform(Image.open(images[idx1]).convert("RGB")).unsqueeze(0).to(device)
            img2 = lpips_transform(Image.open(images[idx2]).convert("RGB")).unsqueeze(0).to(device)

            with torch.no_grad():
                val = lpips_metric(img1, img2)
            current_scores.append(val.item())

        class_avg = sum(current_scores) / len(current_scores)
        all_scores.append(class_avg)
        print(f"Class {class_folder} Diversity: {class_avg:.4f}")

    avg_diversity = sum(all_scores) / len(all_scores) if all_scores else 0.0
    return avg_diversity

In [15]:

print(f"Using device: {DEVICE}")
print(f"Evaluation Target Resolution: {IMAGE_SIZE}x{IMAGE_SIZE}")

fid = FrechetInceptionDistance(feature=2048, normalize=True).to(DEVICE)
inception_score = InceptionScore(feature=2048, normalize=True).to(DEVICE)

print("\n[1/4] Processing Real Images (Resizing to 32x32)...")
real_loader = get_dataloader(REAL_DATA_DIR, BATCH_SIZE, IMAGE_SIZE, is_real_data=True)

for batch in tqdm(real_loader):
    batch = batch.to(DEVICE)
    fid.update(batch, real=True)

print("\n[2/4] Processing Generated Images...")
fake_loader = get_dataloader(GEN_DATA_DIR, BATCH_SIZE, IMAGE_SIZE, is_real_data=False)

for batch in tqdm(fake_loader):
    batch = batch.to(DEVICE)
    fid.update(batch, real=False)
    inception_score.update(batch)

print("\n[3/4] Computing Global Metrics...")

try:
    fid_value = fid.compute()
    print(f">>> Global FID Score: {fid_value.item():.4f} (Lower is better)")
except Exception as e:
    print(f"Error computing FID: {e}")

try:
    is_mean, is_std = inception_score.compute()
    print(f">>> Global Inception Score: {is_mean.item():.4f} +/- {is_std.item():.4f} (Higher is better)")
except Exception as e:
    print(f"Error computing IS: {e}")

print("\n[4/4] Computing LPIPS Diversity...")
try:
    lpips_val = calculate_lpips_diversity(GEN_DATA_DIR, DEVICE)
    print(f">>> Average Per-Class LPIPS Diversity: {lpips_val:.4f} (Higher is better for diversity)")
except Exception as e:
    print(f"Error computing LPIPS: {e}")

fid.reset()
inception_score.reset()



Using device: cuda
Evaluation Target Resolution: 32x32


/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)



[1/4] Processing Real Images (Resizing to 32x32)...
Found 10331 images in /content/train


100%|██████████| 207/207 [00:06<00:00, 29.77it/s]



[2/4] Processing Generated Images...
Found 50000 images in /content/generated


100%|██████████| 1000/1000 [00:49<00:00, 20.20it/s]



[3/4] Computing Global Metrics...
>>> Global FID Score: 8.9294 (Lower is better)
>>> Global Inception Score: 1.0652 +/- 0.0010 (Higher is better)

[4/4] Computing LPIPS Diversity...

=== Calculating Per-Class LPIPS Diversity (Sampling 1000 pairs) ===
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 244MB/s]


Class 0000 Diversity: 0.4428
Class 0001 Diversity: 0.4713
Class 0002 Diversity: 0.4219
Class 0003 Diversity: 0.5008
Class 0004 Diversity: 0.4993
Class 0005 Diversity: 0.5232
Class 0006 Diversity: 0.4697
Class 0007 Diversity: 0.4485
Class 0008 Diversity: 0.4373
Class 0009 Diversity: 0.5039
>>> Average Per-Class LPIPS Diversity: 0.4719 (Higher is better for diversity)
